# Ireland's Population Analysis
The aim of this notebook is to analyse the sexes difference by age in Ireland.

The analysis in this notebook is comprised of 3 parts:
* Part 1 - Here it will be analysed the difference between the sexes by ages in Ireland. Firstly, we will calculate the weighted mean of the sexes by age in Ireland. Then, we will compare weighted means of males vs females to see how they differ
* Part 2 - In the second part, we will group the people within 5 years from 35 into one age group. And then calculate the population difference between the sexes in that age group. 
* Part 3 - Here we will figure out in which Ireland's county there is the largest weighted mean difference in age between sexes.  

## Part 1 Analyses of the difference between the sexes by ages in Ireland

In [21]:
import pandas as pd # Working with dataframes

In [22]:
# Define the source link from which you will get the data
url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/FY006A/CSV/1.0/en"
# Use the link to get the data
df = pd.read_csv(url)
df.head()

,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
0,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,IE0,Ireland,Number,5149139
1,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-1492-13a3-e055-000000000001,Carlow County Council,Number,61968
2,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-1433-13a3-e055-000000000001,Dublin City Council,Number,592713
3,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-149f-13a3-e055-000000000001,Dún Laoghaire Rathdown County Council,Number,233860
4,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-14a0-13a3-e055-000000000001,Fingal County Council,Number,330506


In [23]:
# Check out the names of the columns
headers = df.columns.tolist()
headers

['STATISTIC',
 'Statistic Label',
 'TLIST(A1)',
 'CensusYear',
 'C02199V02655',
 'Sex',
 'C02076V03371',
 'Single Year of Age',
 'C03789V04537',
 'Administrative Counties',
 'UNIT',
 'VALUE']

The df has many columns but we will keep just the following four columns for further analysis: 'Sex', 'Single Year of Age', "VALUE", "Administrative Counties".

In [24]:
# Define the columns to keep to make the df cleaner
to_keep = ['Sex', 'Single Year of Age', "VALUE", "Administrative Counties"]

# Keep the Sex, VALUE, Administrative Counties and the Single Year of Age columns for further analysis
df = df[to_keep]
df.head()

,Sex,Single Year of Age,VALUE,Administrative Counties
0,Both sexes,All ages,5149139,Ireland
1,Both sexes,All ages,61968,Carlow County Council
2,Both sexes,All ages,592713,Dublin City Council
3,Both sexes,All ages,233860,Dún Laoghaire Rathdown County Council
4,Both sexes,All ages,330506,Fingal County Council


In [25]:
df["Single Year of Age"].unique()

array(['All ages', 'Under 1 year', '1 year', '2 years', '3 years',
       '4 years', '5 years', '6 years', '7 years', '8 years', '9 years',
       '10 years', '11 years', '12 years', '13 years', '14 years',
       '15 years', '16 years', '17 years', '18 years', '19 years',
       '20 years', '21 years', '22 years', '23 years', '24 years',
       '25 years', '26 years', '27 years', '28 years', '29 years',
       '30 years', '31 years', '32 years', '33 years', '34 years',
       '35 years', '36 years', '37 years', '38 years', '39 years',
       '40 years', '41 years', '42 years', '43 years', '44 years',
       '45 years', '46 years', '47 years', '48 years', '49 years',
       '50 years', '51 years', '52 years', '53 years', '54 years',
       '55 years', '56 years', '57 years', '58 years', '59 years',
       '60 years', '61 years', '62 years', '63 years', '64 years',
       '65 years', '66 years', '67 years', '68 years', '69 years',
       '70 years', '71 years', '72 years', '73 years', '

The Single Year of Age column has non-digit values, although, we need just digit values. Hence, we will remove all rows with 'All ages', we will replace 'under 1 year' with 0 years and we will remove any non-digit value from a this row.

In [26]:
# Clean the Single Year of Age column. Remove non-digit values and set the type to int64 for further analysis
df['Single Year of Age'] = df['Single Year of Age'].str.replace('Under 1 year', '0')
# Remove the rows where Single Year of Age == All ages
df = df[df["Single Year of Age"] != "All ages"]
df['Single Year of Age'] = df['Single Year of Age'].str.replace('\\D', '', regex=True)
# df = df[df["Single Year of Age"] != ""]
df['Single Year of Age'] = df['Single Year of Age'].astype('int64')

In [27]:
# Check the unique values of the Sex column
df["Sex"].unique()

array(['Both sexes', 'Male', 'Female'], dtype=object)

In [28]:
# Keep only the rows where Sex == Female or Male
df = df[df["Sex"] != "Both sexes"]

In [29]:
df.head()

,Sex,Single Year of Age,VALUE,Administrative Counties
3296,Male,0,29610,Ireland
3297,Male,0,346,Carlow County Council
3298,Male,0,3188,Dublin City Council
3299,Male,0,1269,Dún Laoghaire Rathdown County Council
3300,Male,0,2059,Fingal County Council


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6464 entries, 3296 to 9791
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Sex                      6464 non-null   object
 1   Single Year of Age       6464 non-null   int64 
 2   VALUE                    6464 non-null   int64 
 3   Administrative Counties  6464 non-null   object
dtypes: int64(2), object(2)
memory usage: 252.5+ KB


### Weighted Mean

Weighted mean is sum(age*population at age) / sum (populations at age)

The column administrative counties is divided into counties in Ireland. Although some rows that are equal to Ireland, represent the total count for all counties. However, we need to get the total number in Ireland but not subdivided by countes. Hence, we will filter rows for a particular county rather than Ireland.

In [31]:
# Keep the rows that are equal to Ireland in the Administrative Counties column
df_anal = df[df["Administrative Counties"] == "Ireland"]
# Convert the df into a pivot table
df_anal = pd.pivot_table(df_anal, values='VALUE',index="Single Year of Age",columns="Sex")

print (df_anal.head(10))
# Write out the entire file to local machine
df_anal.to_csv("population_for_analysis.csv")

Sex                  Female     Male
Single Year of Age                  
0                   28186.0  29610.0
1                   27545.0  28875.0
2                   28974.0  30236.0
3                   29483.0  31001.0
4                   29819.0  31686.0
5                   31342.0  32684.0
6                   32622.0  34092.0
7                   33572.0  35429.0
8                   34437.0  36296.0
9                   35227.0  36969.0


### The difference between the sexes by age

In [32]:
def WeightedMean(gender: str, df: 'pd.DataFrame'):
    """
    Calculate the weighted mean age for a specific gender group.

    The weighted mean gives the true average age of the population,
    where each age is weighted by the number of people in that age group.
    """

    # Step 1: Calculate total population for the selected gender
    # Sums all values in the specified gender column
    number_people = df[gender].sum()

    # Step 2: Compute Σ(age × population_at_age)
    # df.index contains ages (0, 1, 2, ...)
    # .mul(df.index, axis=0) multiplies each population by its corresponding age
    # .sum() gives total "person-years" of age
    cumages = df[gender].mul(df.index, axis=0).sum()

    # Step 3: Calculate the weighted mean 
    weighted_mean = cumages/number_people

    return weighted_mean

In [33]:
# Calculate the weighted male age
weighted_male_age = WeightedMean("Male", df_anal)
# Calculate the weighted female age
weighted_female_age = WeightedMean("Female", df_anal)
# Calculate the weighted mean difference between sexes in Ireland
difference = abs(weighted_male_age - weighted_female_age)
difference

np.float64(1.2003481616747962)

## Part 2 the population difference between the sexes for the age group 30-40

In [34]:
# Define the age group 30-40
variable = 35
lower_val = variable - 5
upper_val = variable + 5

In [35]:
# Subset the df to have only rows where age is equal to a value within a defined above age group
df_subset = df_anal.iloc[lower_val:upper_val+1]

In [36]:
# Calculate the weighted male age
weighted_male_age = WeightedMean("Male", df_subset)
# Calculate the weighted female age
weighted_female_age = WeightedMean("Female", df_subset)
# Calculate the weighted mean difference between sexes in Ireland for the age group defined above
difference = weighted_male_age - weighted_female_age
abs(difference)

np.float64(0.025187922337899238)

## Part 3 Region's in Ireland has the biggest population difference between the sexes in that age group

In this task, we will figure out in which Ireland's county there is the largest weighted mean difference in age between sexes. 

In [37]:
# Keep rows that are not equal to Ireland to calculate sexes difference by age in different counties
df_county = df[df["Administrative Counties"] != "Ireland"]
# Get the names of all counties
counties_ireland = df_county["Administrative Counties"].unique()
# Create a pivot table 
df_county = pd.pivot_table(df_county, values='VALUE', index="Single Year of Age", columns=["Administrative Counties", "Sex"])

df_county.head()

Administrative Counties Carlow County Council        Cavan County Council  \
Sex                                    Female   Male               Female   
Single Year of Age                                                          
0                                       353.0  346.0                501.0   
1                                       302.0  347.0                477.0   
2                                       334.0  355.0                520.0   
3                                       378.0  376.0                525.0   
4                                       369.0  376.0                492.0   

Administrative Counties        Clare County Council        Cork City Council  \
Sex                       Male               Female   Male            Female   
Single Year of Age                                                             
0                        505.0                691.0  686.0            1124.0   
1                        527.0                704.0  656.0            1136.0   
2                        541.0                744.0  729.0            1162.0   
3                        560.0                745.0  723.0            1106.0   
4                        555.0                766.0  739.0            1101.0   

Administrative Counties         Cork County Council          ...  \
Sex                        Male              Female    Male  ...   
Single Year of Age                                           ...   
0                        1159.0              2055.0  2135.0  ...   
1                        1148.0              2045.0  2070.0  ...   
2                        1144.0              2106.0  2184.0  ...   
3                        1188.0              2177.0  2260.0  ...   
4                        1142.0              2262.0  2358.0  ...   

Administrative Counties Tipperary County Council          \
Sex                                       Female    Male   
Single Year of Age                                         
0                                          915.0  1017.0   
1                                          872.0   936.0   
2                                          879.0   957.0   
3                                         1019.0  1029.0   
4                                          998.0  1074.0   

Administrative Counties Waterford City & County Council         \
Sex                                              Female   Male   
Single Year of Age                                               
0                                                 662.0  678.0   
1                                                 602.0  664.0   
2                                                 665.0  664.0   
3                                                 721.0  755.0   
4                                                 634.0  756.0   

Administrative Counties Westmeath County Council         \
Sex                                       Female   Male   
Single Year of Age                                        
0                                          535.0  594.0   
1                                          602.0  556.0   
2                                          532.0  576.0   
3                                          558.0  590.0   
4                                          609.0  599.0   

Administrative Counties Wexford County Council         Wicklow County Council  \
Sex                                     Female    Male                 Female   
Single Year of Age                                                              
0                                        882.0   910.0                  800.0   
1                                        878.0   846.0                  820.0   
2                                        921.0   914.0                  890.0   
3                                        903.0   961.0                  886.0   
4                                        921.0  1020.0                  887.0   

Administrative Counties          
Sex                        Male

In [38]:
def largest_diff(county_df: 'pd.DataFrame'):
    """
    Calculate the **absolute difference** in weighted mean age between males and females
    in a given county.

    This function measures gender age gap — how much older (or younger) one gender is
    compared to the other, on average.
    """
    # Step 1: Compute weighted mean age for males
    weighted_male_age = WeightedMean("Male", county_df)
    # Step 2: Compute weighted mean age for females
    weighted_female_age = WeightedMean("Female", county_df)
    # Step 3: Calculate raw difference (male - female)
    difference = weighted_male_age - weighted_female_age

    # Step 4: Return absolute value of age weighted mean difference between sexes 
    return abs(difference)

In [39]:
# Create an empty dictionary that you could save the age weighted mean differences between sexes for each county 
diff_dict = {}

# Compute the age weighted mean difference between sexes for each county Ireland
# and store those results in dictionary where the key is the name of the county
# and the value is the difference
for county in counties_ireland:
    # Calculate the difference
    diff = largest_diff(df_county[county])
    # Add the difference to a dictionary
    diff_dict[county] = diff

    
diff_dict

{'Carlow County Council': np.float64(1.0278457155090948),
 'Dublin City Council': np.float64(1.4813908083433844),
 'Dún Laoghaire Rathdown County Council': np.float64(2.1155658855442496),
 'Fingal County Council': np.float64(1.3615402725255592),
 'South Dublin County Council': np.float64(1.6665821401583045),
 'Kildare County Council': np.float64(1.0071409409568801),
 'Kilkenny County Council': np.float64(1.1738997359885772),
 'Laois County Council': np.float64(0.731747944549916),
 'Longford County Council': np.float64(0.9462884147368413),
 'Louth County Council': np.float64(1.2994206644849982),
 'Meath County Council': np.float64(0.9425363226339414),
 'Offaly County Council': np.float64(1.1343899122394063),
 'Westmeath County Council': np.float64(1.0760278435416168),
 'Wexford County Council': np.float64(1.0588603988827003),
 'Wicklow County Council': np.float64(1.2962588220643312),
 'Clare County Council': np.float64(0.7929128664809397),
 'Cork City Council': np.float64(1.362115596141

In [40]:
# Use list cmprehension to loop through the differences stored in the dictionary
# and find which county or counties have the largest age weighted mean difference between sexes
max_entries = [(k, v) for k, v in diff_dict.items() 
               if v == max(diff_dict.values())]

max_entries

[('Dún Laoghaire Rathdown County Council', np.float64(2.1155658855442496))]